# Code used to create the dataset for the second HERMES data challenge

In [ ]:
providers= ['The European Film Gateway', 'EUscreen']

In [ ]:
import requests
import json
import os
import cv2

# API_KEY = <the API key>

API_ENDPOINT = 'https://api.europeana.eu/record/v2/search.json'
QUERY_PARAMS = {
    'wskey': API_KEY,
    'query': 'provider_aggregation_edm_isShownBy:*mp4*', 
    'qf': ['PROVIDER:"EUscreen"', 'TYPE:VIDEO'], #replaced once more for downloading video and metadata from 'The European Film Gateway'
    'reusability': 'open',
    'rows': 10,  # Number of results per request
    'profile': 'rich',
    'cursor': '*'
}

OUTPUT_DIR = './data/europeana_nl' # replaced by the name of the directory to save data and metadata from 'The European Film Gateway'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def download_video(url, filename):
    """
    Download video from URL and save to filename.
    """
    print(f"Downloading: {url} -> {filename}")
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024):
                f.write(chunk)
        print(f"Video saved: {filename}")
    else:
        print(f"Failed to download video: {url} (Status Code: {response.status_code})")

def save_metadata(metadata, filename):
    """
    Save metadata to a JSON file.
    """
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=4)
    print(f"Metadata saved: {filename}")

def main():
    cursor = '*'
    while cursor:
        print(f"\nQuerying API with cursor: {cursor}")
        
        QUERY_PARAMS['cursor'] = cursor
        response = requests.get(API_ENDPOINT, params=QUERY_PARAMS)
        
        if response.status_code != 200:
            print(f"API request failed! Status Code: {response.status_code}")
            print("Response:", response.text)
            break

        data = response.json()
        print(f"API Response: {json.dumps(data, indent=2)[:500]} ...")

        if 'items' not in data or not data['items']:
            print("No items found. Stopping.")
            break

        for item in data['items']:
            if 'edmIsShownBy' in item:
                video_url = item['edmIsShownBy']
                if isinstance(video_url, list):
                    video_url = video_url[0]
                title = item.get('title', ['untitled'])[0].replace('/', '_')
                video_filename = os.path.join(OUTPUT_DIR, f'{title}.mp4')
                metadata_filename = os.path.join(OUTPUT_DIR, f'{title}.json')
                download_video(video_url, video_filename)
                save_metadata(item, metadata_filename)
            else:
                print(f"No video URL found for item: {item.get('title', 'Unknown Title')}")

        # Correctly update the cursor after processing items
        cursor = data.get('nextCursor', None)
        if cursor:
            print(f"Moving to next page with cursor: {cursor}")
        else:
            print("No more pages. Stopping.")
            break

if __name__ == '__main__':
    main()

In [ ]:
import os
import random

def create_random_file_list (directory):
    """
    Create a list of files in the data directory in a random order.
    """
    files = os.listdir(directory)
    mp4_files = [file for file in files if file.endswith('.mp4')]
    random.shuffle(mp4_files)
    return mp4_files

In [ ]:
dff_dir= './data/europeana_dff'
nl_dir= './data/europeana_nl'

In [ ]:
dff_file_list= create_random_file_list (dff_dir)
dff_file_list

In [ ]:
nl_file_list= create_random_file_list (nl_dir)
nl_file_list

In [ ]:
import os
import shutil
import cv2
import pandas as pd

def create_20_hour_dataset (file_list, source_dir, destination_dir):
    """
    Copy 20 hours of video data to a new directory.
    """
    def create_10_hour_list (file_list, source_dir):
        total_duration= 0
        max_duration= 72000
        selected_vids= []
        for filename in file_list: 
            video_path = os.path.join(source_dir, filename)
            try:
                cap = cv2.VideoCapture(video_path)
                total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                fps = cap.get(cv2.CAP_PROP_FPS)
                video_duration = total_frames / fps
                total_duration+= video_duration
                cap.release()
                if total_duration < max_duration:
                    selected_vids.append(filename)
                else: 
                    break
            except Exception as e:
                print(f"Error processing {filename}: {e}")
        return selected_vids

    select_list= create_10_hour_list (file_list, source_dir)
    os.makedirs(destination_dir, exist_ok=True)
    
    for file_name in select_list: 
        base_name = os.path.splitext(file_name)[0]
        mp4_path = os.path.join(source_dir, f"{base_name}.mp4")
        json_path = os.path.join(source_dir, f"{base_name}.json")
        shutil.copy2(mp4_path, destination_dir)
        shutil.copy2(json_path, destination_dir)

    print(f" Finished copying desired files to {destination_dir}.")

In [ ]:
dff_destination_dir= './data/20_hour_dff'
nl_destination_dir= './data/20_hour_nl'

In [ ]:
create_20_hour_dataset (dff_file_list, dff_dir, dff_destination_dir)

In [ ]:
create_20_hour_dataset (nl_file_list, nl_dir, nl_destination_dir)